# Surveillance des Feux de Forêt et Sévérité des Brûlés

## Introduction
Les incendies de forêt constituent une menace majeure pour la biodiversité et le climat dans le bassin du Congo. Ce notebook utilise l'imagerie infrarouge pour détecter les zones activement brûlées et évaluer la sévérité des dommages causés à l'écosystème.

## Objectifs
*   **Identification des brûlés** : Isoler les signatures de charbon et de sol calciné.
*   **Calcul de sévérité (dNBR)** : Quantifier l'impact du feu sur la biomasse.
*   **Inventaire spatial** : Cartographier les cicatrices d'incendies sur la zone d'étude.

## Méthodologie
1.  **Setup** : Initialisation des outils d'analyse.
2.  **Acquisition** : Images Sentinel-2 (Bandes NIR et SWIR2).
3.  **Analyse NBR** : Application du Normalized Burn Ratio.
4.  **Zonage** : Classification des niveaux de brûlage.

In [ ]:
# ====================================================
# ÉTAPE 1 : Configuration
# ====================================================
!pip install geemap earthengine-api rasterio matplotlib -q

import ee, geemap, rasterio
import numpy as np
import matplotlib.pyplot as plt

try: ee.Initialize()
except: 
    ee.Authenticate()
    ee.Initialize(project='geocongoai-api')

print('✅ Système prêt')

## Zone d'Étude (ROI)
Focus régional pour le monitoring des incendies.

In [ ]:
# ====================================================
# ÉTAPE 2 : Définition de la ROI
# ====================================================
roi = ee.Geometry.Rectangle([15.0, -5.0, 16.0, -4.0])

Map = geemap.Map(basemap='Esri.WorldImagery')
Map.centerObject(roi, 8)
Map.addLayer(roi, {'color': 'red'}, 'Zone d'étude')
Map

## Acquisition des Données
Le NBR utilise principalement le Proche Infrarouge (B8) et l'Infrarouge à ondes courtes (B12).

In [ ]:
# ====================================================
# ÉTAPE 3 : Acquisition satellite
# ====================================================
image = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
         .filterBounds(roi).filterDate('2023-01-01', '2023-12-31')
         .median().clip(roi))

geemap.ee_export_image(image.select(['B8', 'B12']), 'fire.tif', scale=30, region=roi)

## Calcul du Normalized Burn Ratio (NBR)
Les zones brûlées ont une forte réflectance dans le SWIR et une faible réflectance dans le NIR. Un indice NBR faible indique une zone brûlée.

In [ ]:
# ====================================================
# ÉTAPE 4 : Algorithme de détection des feux
# ====================================================
with rasterio.open('fire.tif') as src: bands = src.read().astype(np.float32)
nir, swir2 = bands[0], bands[1]
nbr = (nir - swir2) / (nir + swir2 + 1e-8)

burned_mask = nbr < 0.1

plt.figure(figsize=(10, 8))
plt.imshow(burned_mask, cmap='YlOrBr')
plt.title('Détection des Zones Brûlées (NBR)')
plt.axis('off')
plt.show()